# 基于 GRPO 的 Qwen3.5 小模型理科推理能力增强研究
## Kaggle Notebook — 双 T4 一键运行

> 点击 **Run All** 即可从头跑完：SFT 预热 → GRPO 主实验 → 消融实验 → 评估 → 结果图表

## ⚙️ 可调参数
修改下方变量后 Run All，无需改动其他 cell。

In [ ]:
# ============================================================
# 可调参数 — 按需修改
# ============================================================

# 模型（二选一）
MODEL_HF = "Qwen/Qwen3.5-0.8B-Instruct"   # 0.8B，显存友好
# MODEL_HF = "Qwen/Qwen3.5-2B-Instruct"   # 2B，效果更好但更慢

# 数据集
DATASET_NAME = "openai/gsm8k"
DATASET_CONFIG = "main"

# SFT 参数
SFT_EPOCHS = 2
SFT_BATCH_SIZE = 4          # T4 16GB 可用 4
SFT_GRAD_ACCUM = 4
SFT_LR = 2e-4
SFT_LORA_R = 16
SFT_MAX_SEQ = 1024
SFT_MAX_SAMPLES = 4000      # None=全部

# GRPO 参数
GRPO_EPOCHS = 2
GRPO_BATCH_SIZE = 2
GRPO_GRAD_ACCUM = 8
GRPO_LR = 5e-5
GRPO_LORA_R = 16
GRPO_NUM_GENERATIONS = 8    # Group Size
GRPO_KL_COEF = 0.04
GRPO_MAX_PROMPT = 512
GRPO_MAX_COMPLETION = 512
GRPO_MAX_SAMPLES = 2000

# 评估参数
EVAL_MAX_SAMPLES = 200
EVAL_DATASET = "gsm8k"       # gsm8k / scibench / all

# 消融实验开关
RUN_ABLATIONS = True         # False 跳过消融，节省时间

# 输出根目录
OUTPUT_ROOT = "/kaggle/working/outputs"

print("✅ 参数加载完成")

## Step 1: 安装依赖

In [ ]:
!pip install -q transformers>=4.47.0 datasets>=3.0.0 accelerate>=1.0.0
!pip install -q peft>=0.13.0 bitsandbytes>=0.44.0 trl>=0.12.0
!pip install -q wandb tensorboard gradio>=5.0.0
!pip install -q numpy pandas matplotlib seaborn tqdm

print("✅ 依赖安装完成")
print("（如出现 RAPIDS/numba 冲突警告可忽略，不影响 PyTorch 训练）")

## Step 2: 环境检查 + 克隆项目

In [ ]:
import os
import sys
import torch
import json
import time
from pathlib import Path

# HF 镜像（国内加速）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# GPU 信息
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}, {props.total_mem / 1024**3:.1f} GB")

# 克隆项目
REPO_URL = "https://github.com/JKpink/term-end-project.git"
PROJECT_DIR = Path("/kaggle/working/term-end-project")

if not PROJECT_DIR.exists():
    !git clone -q {REPO_URL} {PROJECT_DIR}
    print("✅ 项目已克隆")
else:
    print("✅ 项目已存在，跳过克隆")

# 进入项目目录
SRC_DIR = PROJECT_DIR / "project" / "src"
os.chdir(PROJECT_DIR / "project")
sys.path.insert(0, str(SRC_DIR))

# 安装项目依赖
!pip install -q -r requirements.txt

# 创建输出目录
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

print(f"\n工作目录: {os.getcwd()}")
print(f"源码目录: {SRC_DIR}")
print(f"输出目录: {OUTPUT_ROOT}")
print("✅ 环境就绪")

---
## Phase 1: SFT 预热训练

QLoRA 监督微调，让 Qwen3.5-0.8B 学会 `<think>...</think>` 推理格式。

In [ ]:
SFT_OUTPUT = f"{OUTPUT_ROOT}/sft_kaggle"

sft_cmd = f"""
python {SRC_DIR}/train_sft.py \
    --model_name {MODEL_HF} \
    --output_dir {SFT_OUTPUT} \
    --num_epochs {SFT_EPOCHS} \
    --per_device_batch_size {SFT_BATCH_SIZE} \
    --gradient_accumulation_steps {SFT_GRAD_ACCUM} \
    --learning_rate {SFT_LR} \
    --lora_r {SFT_LORA_R} \
    --max_seq_length {SFT_MAX_SEQ} \
    {"--max_train_samples " + str(SFT_MAX_SAMPLES) if SFT_MAX_SAMPLES else ""} \
    --logging_steps 10 \
    --save_steps 200 \
    --use_flash_attention
"""

print(f"SFT 输出目录: {SFT_OUTPUT}")
print(f"模型: {MODEL_HF}")
print(f"Epochs: {SFT_EPOCHS}, Batch: {SFT_BATCH_SIZE}, Grad Accum: {SFT_GRAD_ACCUM}")
print(f"有效 Batch Size: {SFT_BATCH_SIZE * SFT_GRAD_ACCUM}")

t0 = time.time()
!{sft_cmd}
t1 = time.time()

SFT_ADAPTER = f"{SFT_OUTPUT}/final"
if os.path.exists(SFT_ADAPTER):
    print(f"\n✅ SFT 完成！耗时 {(t1-t0)/60:.1f} 分钟")
    print(f"适配器保存至: {SFT_ADAPTER}")
else:
    print("\n❌ SFT 失败，请检查上方日志")
    SFT_ADAPTER = None

---
## Phase 2: GRPO 主实验

核心方法 — SFT 适配器初始化 + GRPO 强化学习。

In [ ]:
GRPO_MAIN_OUTPUT = f"{OUTPUT_ROOT}/grpo_kaggle_main"

if SFT_ADAPTER and os.path.exists(SFT_ADAPTER):
    grpo_cmd = f"""
    python {SRC_DIR}/train_grpo.py \
        --model_name {MODEL_HF} \
        --sft_adapter_path {SFT_ADAPTER} \
        --output_dir {GRPO_MAIN_OUTPUT} \
        --reward_type full \
        --num_generations {GRPO_NUM_GENERATIONS} \
        --kl_coef {GRPO_KL_COEF} \
        --learning_rate {GRPO_LR} \
        --lora_r {GRPO_LORA_R} \
        --num_train_epochs {GRPO_EPOCHS} \
        --per_device_batch_size {GRPO_BATCH_SIZE} \
        --gradient_accumulation_steps {GRPO_GRAD_ACCUM} \
        --max_prompt_length {GRPO_MAX_PROMPT} \
        --max_completion_length {GRPO_MAX_COMPLETION} \
        {"--max_train_samples " + str(GRPO_MAX_SAMPLES) if GRPO_MAX_SAMPLES else ""} \
        --logging_steps 5 \
        --save_steps 100
    """

    print(f"GRPO 输出目录: {GRPO_MAIN_OUTPUT}")
    print(f"SFT 适配器: {SFT_ADAPTER}")
    print(f"Group Size: {GRPO_NUM_GENERATIONS}, KL Coef: {GRPO_KL_COEF}")

    t0 = time.time()
    !{grpo_cmd}
    t1 = time.time()

    GRPO_ADAPTER = f"{GRPO_MAIN_OUTPUT}/final_grpo"
    if os.path.exists(GRPO_ADAPTER):
        print(f"\n✅ GRPO 主实验完成！耗时 {(t1-t0)/60:.1f} 分钟")
    else:
        print("\n❌ GRPO 失败，请检查上方日志")
        GRPO_ADAPTER = None
else:
    print("❌ SFT 适配器不存在，跳过 GRPO 主实验")
    GRPO_ADAPTER = None

---
## Phase 3: 消融实验

5 组消融并行/串行（每组 try/except 保护，失败不阻断后续）。

In [ ]:
if not RUN_ABLATIONS:
    print("⏭️  已跳过消融实验（RUN_ABLATIONS=False）")
else:
    # 消融配置: (名称, reward_type, use_sft, num_generations, kl_coef)
    ablations = [
        ("abl_no_sft",      "full",             False, GRPO_NUM_GENERATIONS, GRPO_KL_COEF),
        ("abl_correctness", "correctness_only",  True,  GRPO_NUM_GENERATIONS, GRPO_KL_COEF),
        ("abl_format",      "correctness_and_format", True, GRPO_NUM_GENERATIONS, GRPO_KL_COEF),
        ("abl_g4",          "full",             True,  4,                      GRPO_KL_COEF),
        ("abl_g16",         "full",             True,  16,                     GRPO_KL_COEF),
        ("abl_kl001",       "full",             True,  GRPO_NUM_GENERATIONS,   0.01),
        ("abl_kl10",        "full",             True,  GRPO_NUM_GENERATIONS,   0.10),
    ]

    ablation_results = {}

    for name, reward_type, use_sft, gsize, kl in ablations:
        print(f"\n{'='*60}")
        print(f"消融实验: {name}")
        print(f"  Reward: {reward_type}, SFT预热: {use_sft}, G={gsize}, KL={kl}")
        print(f"{'='*60}")

        try:
            output_dir = f"{OUTPUT_ROOT}/grpo_kaggle_{name}"

            cmd_parts = [
                f"python {SRC_DIR}/train_grpo.py",
                f"--model_name {MODEL_HF}",
                f"--output_dir {output_dir}",
                f"--reward_type {reward_type}",
                f"--num_generations {gsize}",
                f"--kl_coef {kl}",
                f"--learning_rate {GRPO_LR}",
                f"--lora_r {GRPO_LORA_R}",
                f"--num_train_epochs 1",  # 消融只跑1轮
                f"--per_device_batch_size {GRPO_BATCH_SIZE}",
                f"--gradient_accumulation_steps {GRPO_GRAD_ACCUM}",
                f"--max_prompt_length {GRPO_MAX_PROMPT}",
                f"--max_completion_length {GRPO_MAX_COMPLETION}",
                f"--max_train_samples 500",  # 消融用500样本
                f"--logging_steps 5",
                f"--save_steps 100",
            ]

            if use_sft and SFT_ADAPTER and os.path.exists(SFT_ADAPTER):
                cmd_parts.append(f"--sft_adapter_path {SFT_ADAPTER}")

            cmd = " \\\n    ".join(cmd_parts)

            t0 = time.time()
            !{cmd}
            t1 = time.time()

            adapter_path = f"{output_dir}/final_grpo"
            if os.path.exists(adapter_path):
                ablation_results[name] = {"status": "ok", "adapter": adapter_path, "time_min": (t1-t0)/60}
                print(f"✅ {name} 完成！耗时 {(t1-t0)/60:.1f} 分钟")
            else:
                ablation_results[name] = {"status": "no_adapter", "adapter": None}
                print(f"⚠️  {name}: 训练完成但未找到适配器")

        except Exception as e:
            ablation_results[name] = {"status": "failed", "error": str(e)[:200]}
            print(f"❌ {name} 失败: {str(e)[:200]}")

    # 保存消融结果摘要
    with open(f"{OUTPUT_ROOT}/ablation_summary.json", "w") as f:
        json.dump(ablation_results, f, indent=2, ensure_ascii=False)

    print(f"\n{'='*60}")
    print("消融实验汇总:")
    for name, info in ablation_results.items():
        status_icon = "✅" if info["status"] == "ok" else "❌"
        print(f"  {status_icon} {name}: {info['status']}")
    print(f"{'='*60}")

---
## Phase 4: 评估所有模型

In [ ]:
eval_tasks = [
    ("base_no_think", MODEL_HF, None, False),
    ("base_think",    MODEL_HF, None, True),
    ("sft",           MODEL_HF, SFT_ADAPTER, True),
    ("grpo_main",     MODEL_HF, GRPO_ADAPTER, True),
]

# 添加成功的消融实验
if RUN_ABLATIONS:
    for name, info in ablation_results.items():
        if info["status"] == "ok" and info["adapter"]:
            eval_tasks.append((name, MODEL_HF, info["adapter"], True))

eval_metrics_all = {}

for label, model_hf, adapter, thinking in eval_tasks:
    if adapter and not os.path.exists(adapter):
        print(f"⚠️  跳过 {label}: 适配器不存在 ({adapter})")
        continue

    print(f"\n{'='*60}")
    print(f"评估: {label}")
    print(f"  Model: {model_hf}")
    print(f"  Adapter: {adapter}")
    print(f"  Thinking: {thinking}")
    print(f"{'='*60}")

    eval_output = f"{OUTPUT_ROOT}/eval_{label}"

    cmd_parts = [
        f"python {SRC_DIR}/evaluate.py",
        f"--model_name {model_hf}",
        f"--dataset {EVAL_DATASET}",
        f"--max_samples {EVAL_MAX_SAMPLES}",
        f"--output_dir {eval_output}",
    ]
    if adapter:
        cmd_parts.append(f"--adapter_path {adapter}")
    if not thinking:
        cmd_parts.append("--no_thinking")
    else:
        cmd_parts.append("--enable_thinking")

    cmd = " \\\n    ".join(cmd_parts)

    try:
        !{cmd}

        # 读取评估指标
        metrics_file = f"{eval_output}/{EVAL_DATASET}_metrics.json"
        if os.path.exists(metrics_file):
            with open(metrics_file) as f:
                metrics = json.load(f)
            eval_metrics_all[label] = metrics
            print(f"  Accuracy: {metrics['accuracy']}%, Thinking Rate: {metrics['thinking_rate']}%, Avg Steps: {metrics['avg_reasoning_steps']}")
        else:
            print(f"  ⚠️  未找到评估指标文件: {metrics_file}")
    except Exception as e:
        print(f"  ❌ 评估失败: {str(e)[:200]}")

# 保存汇总
with open(f"{OUTPUT_ROOT}/all_eval_metrics.json", "w") as f:
    json.dump(eval_metrics_all, f, indent=2, ensure_ascii=False)

print(f"\n✅ 评估完成，结果保存至 {OUTPUT_ROOT}/all_eval_metrics.json")

---
## Phase 5: 结果汇总

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if eval_metrics_all:
    # 构建汇总表
    rows = []
    for label, m in eval_metrics_all.items():
        rows.append({
            "方法": label,
            "准确率 (%)": m["accuracy"],
            "思考率 (%)": m["thinking_rate"],
            "平均推理步数": m["avg_reasoning_steps"],
        })

    df = pd.DataFrame(rows)
    df = df.sort_values("准确率 (%)", ascending=False)

    print("\n" + "="*70)
    print("📊 实验结果汇总")
    print("="*70)
    print(df.to_string(index=False))

    # 柱状图
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 准确率
    labels = df["方法"].values
    acc_values = df["准确率 (%)"].values
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(labels)))

    axes[0].barh(labels, acc_values, color=colors)
    axes[0].set_xlabel("Accuracy (%)")
    axes[0].set_title("GSM8K Accuracy Comparison")
    axes[0].invert_yaxis()
    for i, v in enumerate(acc_values):
        axes[0].text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=10)

    # 思考率
    think_values = df["思考率 (%)"].values
    axes[1].barh(labels, think_values, color=colors)
    axes[1].set_xlabel("Thinking Rate (%)")
    axes[1].set_title("Thinking Format Compliance")
    axes[1].invert_yaxis()
    for i, v in enumerate(think_values):
        axes[1].text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=10)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_ROOT}/results_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\n📈 图表保存至: {OUTPUT_ROOT}/results_comparison.png")
else:
    print("⚠️  无评估数据，跳过图表生成")

---
## 🎉 全部完成！

### 产出文件

```
/kaggle/working/outputs/
├── sft_kaggle/                SFT 适配器
├── grpo_kaggle_main/          GRPO 主实验模型
├── grpo_kaggle_abl_*/         消融实验模型
├── eval_*/                    评估结果 (CSV + JSON)
├── all_eval_metrics.json      汇总指标
├── ablation_summary.json      消融汇总
└── results_comparison.png     对比柱状图
```

### 下一步

- 下载 `results_comparison.png` 用于论文
- 下载 `all_eval_metrics.json` 获取完整评估数据
- 在 Kaggle 右侧面板 → **Save Version** 保存所有产出